## Import necessary dependencies


In [4]:

!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard gradio
!pip install jinja2
! pip install ipywidgets
! jupyter nbextension enable --py widgetsnbextension
! pip install --upgrade datasets

  Using cached transformers-4.55.2-py3-none-any.whl.metadata (41 kB)
  Using cached accelerate-1.10.0-py3-none-any.whl.metadata (19 kB)
  Using cached evaluate-0.4.5-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached gradio-5.42.0-py3-none-any.whl.metadata (16 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached torchcodec-0.6.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached torch-2.8.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached rapidfuzz-3.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached Brotli-1.1.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached gradio_

In [5]:
from datasets import load_dataset, DatasetDict
from transformers import (
    WhisperTokenizer,
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

In [6]:
from datasets import Audio
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch
import evaluate


# Using whisper-tiny model (has 39M parameters)

In [7]:
model_id = 'openai/whisper-small'
out_dir = 'whisper_tiny_np'
epochs = 50
batch_size = 4

In [8]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language='Nepali', task='transcribe')
processor = WhisperProcessor.from_pretrained(model_id, language='Nepali', task='transcribe')


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [9]:

training_np = load_dataset("fsicoli/common_voice_19_0", "ne-NP", split="train", trust_remote_code=True)
validation_np = load_dataset("fsicoli/common_voice_19_0", "ne-NP", split="test", trust_remote_code=True)

README.md: 0.00B [00:00, ?B/s]

common_voice_19_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

n_shards.json: 0.00B [00:00, ?B/s]

...


Reading metadata...: 205it [00:00, 81943.42it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 638it [00:00, 95614.61it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 67it [00:00, 74838.45it/s]


In [10]:

# training_np = load_dataset("spktsagar/openslr-nepali-asr-cleaned",  split="train", trust_remote_code=True)
# validation_np = load_dataset("spktsagar/openslr-nepali-asr-cleaned",  split="test", trust_remote_code=True)

In [11]:
training_np[4]

{'client_id': '9f8a47cee5574b287a8f93f5498d81115cf1dfbd718ead4f2265e4400f7de0f017a58a2c8c1245e0d3ceeccffa5b110322c4f784aa8a9785e3219557cb44395e',
 'path': '/root/.cache/huggingface/datasets/downloads/extracted/2078d4f647abb87146c4e6361776aff17e038b4472a795fd02ab22d7c2574c59/ne-NP_train_0/common_voice_ne-NP_35314093.mp3',
 'audio': {'path': '/root/.cache/huggingface/datasets/downloads/extracted/2078d4f647abb87146c4e6361776aff17e038b4472a795fd02ab22d7c2574c59/ne-NP_train_0/common_voice_ne-NP_35314093.mp3',
  'array': array([ 4.54747351e-13,  1.42108547e-12,  2.27373675e-13, ...,
         -1.78488481e-05, -2.26100929e-05, -1.12088283e-05]),
  'sampling_rate': 48000},
 'sentence': 'हैन, जर्मनी समुह चरणबाट बहिरिने सम्भावना धेरै देख्छु म त ।',
 'up_votes': 4,
 'down_votes': 0,
 'age': 'thirties',
 'gender': 'male_masculine',
 'accent': 'nepali',
 'locale': 'ne-NP',
 'segment': '',
 'variant': ''}

Resmapling at 16khz

In [12]:
training_np = training_np.cast_column('audio', Audio(sampling_rate=16000))
validation_np = validation_np.cast_column('audio', Audio(sampling_rate=16000))

In [13]:
training_np = training_np.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])
validation_np = validation_np.remove_columns(["accent", "age", "client_id", "down_votes", "gender", "locale", "path", "segment", "up_votes"])

In [14]:
for i, data in enumerate(training_np):
  if not 'sentence' in data.keys() or not 'audio' in data.keys():
    print(i, 'not found')

In [15]:
def prepare_dataset(batch):
  audio = batch['audio']
  batch['input_features'] = feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
  batch['labels'] =  tokenizer(batch['sentence']).input_ids
  return batch


training_np = training_np.map(
    prepare_dataset,
    num_proc=1
)

validation_np = validation_np.map(
    prepare_dataset,
    num_proc=1
)

Map:   0%|          | 0/381 [00:00<?, ? examples/s]

Map:   0%|          | 0/205 [00:00<?, ? examples/s]

In [16]:
training_np[0].keys(), validation_np[0].keys()

(dict_keys(['audio', 'sentence', 'variant', 'input_features', 'labels']),
 dict_keys(['audio', 'sentence', 'variant', 'input_features', 'labels']))

In [17]:
input_str = training_np[4]["sentence"]
labels = tokenizer(input_str).input_ids
decoded_with_special = tokenizer.decode(labels, skip_special_tokens=False)
decoded_str = tokenizer.decode(labels, skip_special_tokens=True)

print(f"Input:                 {input_str}")
print(f"Decoded w/ special:    {decoded_with_special}")
print(f"Decoded w/out special: {decoded_str}")
print(f"Are equal:             {input_str == decoded_str}")


Input:                 हैन, जर्मनी समुह चरणबाट बहिरिने सम्भावना धेरै देख्छु म त ।
Decoded w/ special:    <|startoftranscript|><|ne|><|transcribe|><|notimestamps|>हैन, जर्मनी समुह चरणबाट बहिरिने सम्भावना धेरै देख्छु म त ।<|endoftext|>
Decoded w/out special: हैन, जर्मनी समुह चरणबाट बहिरिने सम्भावना धेरै देख्छु म त ।
Are equal:             True


### Preparing the Model


In [18]:
model = WhisperForConditionalGeneration.from_pretrained(model_id)

model.generation_config.task = 'transcribe'
model.generation_config.language = 'nepali'
model.generation_config.forced_decoder_ids = None

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [19]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


In [20]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

## Defining evaluation metrices (word error rate)

In [21]:
metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {'wer': wer}

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir=out_dir,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=1,
    learning_rate=0.00001,
    warmup_steps=500,
    bf16=False,
    fp16=True,
    num_train_epochs=epochs,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=225,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    dataloader_num_workers=2,
    save_total_limit=2,
    lr_scheduler_type='constant',
    seed=42,
    data_seed=42
)

In [23]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=training_np,
    eval_dataset=validation_np,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

/tmp/ipykernel_36/1335044146.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Wer
1,0.742900,0.560032,72.915073
2,0.307300,0.510562,68.630451
3,0.145100,0.557180,69.319051
4,0.082500,0.550684,65.034430
5,0.046900,0.582780,67.788829
6,0.031900,0.641847,66.335119
7,0.029000,0.661191,66.488141
8,0.024600,0.675273,64.498852
9,0.014200,0.743036,64.651875
10,0.016400,0.753909,61.820964


You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3465: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 2

In [ ]:
dataset = load_dataset("spktsagar/openslr-nepali-asr-cleaned", name="cleaned", split='train')

In [ ]:
dataset[0]

### Inference

In [ ]:
from torch.utils.data import DataLoader
dataloader = DataLoader(validation_np, batch_size=4, collate_fn=data_collator)

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:
# Iterate through batches and get model predictions
for batch in dataloader:
    input_features = batch["input_features"]
    labels = batch["labels"]

    # Perform inference (using no_grad for evaluation)
    with torch.no_grad():
        generated_ids = model.generate(input_features, language='ne')

    # Decode the predicted token IDs into text
    predictions = processor.batch_decode(generated_ids, skip_special_tokens=True)
    sents = processor.batch_decode(labels, skip_special_tokens=True)
    # Print or store predictions
    for pred, sen in zip(predictions, sents):
        print(f'GT:{sen}.......... Pred: {pred}')
        
    break

In [ ]:

# Save best model & processor to a clean folder, and record the path
import os, shutil, json, time
from pathlib import Path

# Safety checks
assert 'trainer' in globals(), "trainer is not defined"
assert 'model' in globals(), "model is not defined"
assert 'processor' in globals(), "processor is not defined"
assert 'out_dir' in globals(), "out_dir is not defined"

# Where to save the final export
timestamp = time.strftime("%Y%m%d-%H%M%S")
export_dir = Path('/kaggle/working')/f"{out_dir}_export_{timestamp}"
export_dir.mkdir(parents=True, exist_ok=True)

# If load_best_model_at_end=True, trainer.model is already the best checkpoint
trainer.model.save_pretrained(export_dir.as_posix())
processor.save_pretrained(export_dir.as_posix())

# Store a simple metadata file and a plain-text path
(Path(export_dir)/'model_path.txt').write_text(export_dir.as_posix())
metadata = {
    "saved_at": timestamp,
    "export_dir": export_dir.as_posix(),
    "base_model": str(globals().get("model_id", "")),
    "language": "ne",
    "task": "transcribe",
}
(Path(export_dir)/'export_meta.json').write_text(json.dumps(metadata, indent=2))

print("Model exported to:", export_dir)
print("Wrote path file:", (Path(export_dir)/'model_path.txt'))

## Zip the Export for Download

In [ ]:
import shutil, os
from pathlib import Path

assert 'export_dir' in globals(), "Run the previous cell first to define export_dir"

zip_path = Path('/kaggle/working')/(Path(export_dir).name + ".zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(base_name=zip_path.as_posix().replace('.zip',''), format='zip', root_dir=export_dir.as_posix())
print("Zipped:", zip_path)

## Training curves and metrics

In [ ]:
# Build a metrics DataFrame from Trainer logs and plot
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

logs = trainer.state.log_history  # list of dicts
df = pd.DataFrame(logs)

# Save raw logs
out_csv = Path('/kaggle/working')/f"{out_dir}_log_history.csv"
df.to_csv(out_csv, index=False)
print("Saved log history CSV:", out_csv)

# Helper to plot a metric over step
def plot_metric(df, metric_key, title):
    sub = df.dropna(subset=[metric_key, "step"])
    if len(sub)==0:
        print(f"No entries for {metric_key}")
        return
    plt.figure()
    plt.plot(sub["step"], sub[metric_key])
    plt.title(title)
    plt.xlabel("Step")
    plt.ylabel(metric_key)
    png = Path('/kaggle/working')/f"{out_dir}_{metric_key}.png"
    plt.savefig(png, bbox_inches='tight')
    plt.show()
    print("Saved plot:", png)

for key, title in [
    ("loss", "Training Loss vs Step"),
    ("eval_loss", "Eval Loss vs Step"),
    ("eval_wer", "Eval WER vs Step"),
    ("learning_rate", "Learning Rate vs Step"),
]:
    plot_metric(df, key, title)

## Batch inference on kaggle

In [ ]:
# Use the exported model+processor to transcribe audio files present in a folder (since Kaggle cannot access a microphone).
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
from datasets import Audio
import torchaudio, torch, os, glob, pandas as pd
from pathlib import Path

AUDIO_DIR = "/kaggle/input/your-audio-folder"  # TODO: set this to  dataset
AUDIO_EXTS = ("*.wav","*.mp3","*.flac","*.m4a")

# Load from the just-exported folder
export_path = export_dir  # from previous save cell
processor = WhisperProcessor.from_pretrained(export_path)
model = WhisperForConditionalGeneration.from_pretrained(export_path).to("cuda" if torch.cuda.is_available() else "cpu")

def load_audio_16k(path):
    wav, sr = torchaudio.load(path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    return wav.squeeze(0).numpy(), 16000

results = []
for ext in AUDIO_EXTS:
    for f in glob.glob(os.path.join(AUDIO_DIR, ext)):
        audio, sr = load_audio_16k(f)
        inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
        input_features = inputs.input_features.to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(
                input_features=input_features,
                language="ne",
                task="transcribe",
                max_length=225
            )
        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        results.append({"file": os.path.basename(f), "transcription": text})

df = pd.DataFrame(results)
from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Batch Inference Results", df)
print("Done.")

## Real tme transcription (for vs code)

In [ ]:
import queue, sys, time, threading
import numpy as np
import sounddevice as sd
import torch, torchaudio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_DIR = r"./whisper_tiny_np_export"  # <- replace with downloaded export folder
LANG = "ne"
CHUNK_SECONDS = 5.0      # seconds per chunk
SAMPLE_RATE = 16000      # model expects 16 kHz
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(MODEL_DIR)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.generation_config.task = "transcribe"

q = queue.Queue()

def audio_callback(indata, frames, time_info, status):
    if status:
        print(status, file=sys.stderr)
    # Convert to mono float32 and enqueue
    mono = indata.mean(axis=1).astype(np.float32)
    q.put(mono.copy())

def resample_audio(waveform, orig_sr, target_sr=16000):
    # waveform: numpy 1D
    wav = torch.from_numpy(waveform).unsqueeze(0)  # [1, T]
    res = torchaudio.functional.resample(wav, orig_sr, target_sr)
    return res.squeeze(0).numpy()

def transcribe_chunk(buffer):
    # buffer: numpy 1D at 16kHz
    inputs = processor(buffer, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    input_features = inputs.input_features.to(DEVICE)
    with torch.no_grad():
        generated_ids = model.generate(
            input_features=input_features,
            language=LANG,
            task="transcribe",
            max_length=225
        )
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text.strip()

def run_realtime():
    print("Listening... speak Nepali. Press Ctrl+C to stop.")
    chunk_size = int(CHUNK_SECONDS * SAMPLE_RATE)
    ring = np.zeros(0, dtype=np.float32)

    # Start the audio stream
    with sd.InputStream(channels=1, samplerate=SAMPLE_RATE, callback=audio_callback):
        try:
            last_print = time.time()
            while True:
                # Collect audio from queue
                while not q.empty():
                    ring = np.concatenate([ring, q.get()])
                # When enough audio for a chunk, process
                if ring.shape[0] >= chunk_size:
                    chunk = ring[:chunk_size]
                    ring = ring[chunk_size:]
                    text = transcribe_chunk(chunk)
                    print(f"[{time.strftime('%H:%M:%S')}] {text}")
                # avoid busy loop
                time.sleep(0.05)
        except KeyboardInterrupt:
            print("\nStopped.")

if __name__ == "__main__":
    run_realtime()